# Notebook 2 · Fine-tuning de Stable Diffusion con LoRA usando `diffusers`

**CADI Deep Learning Conceptos** · Docente: Nidia Stella García Roa
**Estudiante:** Ruby Dayana Cárdenas Gómez · Universidad de Cundinamarca · Noviembre 2026

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RubyDayana/modelos-generativos-gan-difusion/blob/main/notebooks/02-StableDiffusion-LoRA-Finetuning.ipynb)

> **Antes de ejecutar:** en Colab ir a *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*. Todo el notebook tarda unos 25-35 minutos (la mayor parte es el entrenamiento).

En el notebook 1 entrené una GAN desde cero. Aquí hago lo contrario: parto de un modelo enorme ya entrenado, **Stable Diffusion v1.5** (≈ 1 066 millones de parámetros), y lo **ajusto** (*fine-tuning*) para que aprenda un estilo nuevo usando muy pocos recursos. Para eso uso la librería **`diffusers`** de Hugging Face y la técnica **LoRA** (*Low-Rank Adaptation*), que entrena menos del 0,2 % de los parámetros.

Estilo objetivo: las fotografías de especímenes de mariposas de la colección del **Smithsonian** (fondo claro, ejemplar centrado y extendido, vista superior). Para que el modelo asocie el estilo con una palabra, uso el disparador inventado **`smthsn`** en todas las descripciones de entrenamiento.

Contenido:
1. Conceptos: difusión, difusión latente, espacio latente y LoRA
2. Instalación y verificación de la GPU
3. Componentes de Stable Diffusion
4. Exploración del espacio latente (VAE) y del proceso de difusión hacia adelante
5. Preparación de los datos
6. Generación antes del fine-tuning
7. Configuración de LoRA y entrenamiento
8. Generación después del fine-tuning y comparación
9. Pruebas: escala de LoRA, métrica CLIP y recarga de los pesos
10. Análisis

## 1. Conceptos clave

**Modelos de difusión (DDPM, Ho et al., 2020).** Se definen dos procesos:

- *Hacia adelante* (fijo): a una imagen limpia $x_0$ se le agrega ruido gaussiano en $T=1000$ pasos hasta convertirla en ruido puro. Gracias a una propiedad de las gaussianas se puede saltar directamente al paso $t$:
$$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon, \qquad \epsilon\sim\mathcal N(0,I)$$
- *Hacia atrás* (aprendido): una red neuronal $\epsilon_\theta(x_t, t)$ aprende a **predecir el ruido** que se agregó. La función de pérdida es simplemente
$$\mathcal L = \mathbb E_{x_0,\epsilon,t}\big[\lVert \epsilon - \epsilon_\theta(x_t,t)\rVert^2\big]$$
Para generar se parte de ruido puro y se va quitando el ruido predicho paso a paso.

**Difusión latente (Rombach et al., 2022).** Hacer difusión directamente sobre imágenes de 512×512×3 es muy costoso. Stable Diffusion usa un **autoencoder variacional (VAE)** que comprime la imagen a un **espacio latente** de 64×64×4 (48 veces menos valores) y hace todo el proceso de difusión ahí. Sus tres piezas:

| Componente | Función |
|---|---|
| **VAE** | codifica la imagen al espacio latente y decodifica el resultado final a píxeles |
| **U-Net** | predice el ruido en el espacio latente; recibe el texto mediante *cross-attention* |
| **Codificador de texto CLIP** | convierte el *prompt* en 77 vectores de 768 dimensiones |

**DALL·E vs. Stable Diffusion.** DALL·E 2 (Ramesh et al., 2022) también usa difusión, pero condicionada en los *embeddings* de imagen de CLIP mediante un modelo *prior* y genera en el espacio de píxeles con etapas de superresolución; es un modelo cerrado. Stable Diffusion trabaja en el espacio latente del VAE y sus pesos son abiertos, lo que permite hacer fine-tuning como en este notebook.

**Guía libre de clasificador (*classifier-free guidance*).** En cada paso se predice el ruido con y sin texto y se combinan: $\hat\epsilon = \epsilon_\varnothing + s(\epsilon_{texto}-\epsilon_\varnothing)$. Con $s=7{,}5$ la imagen sigue más de cerca el prompt.

**LoRA (Hu et al., 2022).** En vez de modificar una matriz de pesos $W\in\mathbb R^{d\times k}$, se congela y se le suma una actualización de rango bajo:
$$W' = W + \tfrac{\alpha}{r}\,BA,\qquad B\in\mathbb R^{d\times r},\; A\in\mathbb R^{r\times k},\; r\ll d$$
Con $r=8$ solo se entrenan unos 1,6 millones de parámetros (0,19 % de la U-Net) en las capas de atención, y el archivo resultante pesa unos 6 MB en lugar de varios GB.

## 2. Instalación y verificación de la GPU

In [ ]:
!pip install -q -U diffusers peft accelerate datasets transformers

In [ ]:
import os, time, json, math, random
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

import diffusers, transformers, peft
print('diffusers', diffusers.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__, '| torch', torch.__version__)

assert torch.cuda.is_available(), 'Activa la GPU: Entorno de ejecución > Cambiar tipo de entorno > GPU T4'
dispositivo = 'cuda'
props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name} | memoria: {props.total_memory/1024**3:.1f} GB')

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
os.makedirs('figuras', exist_ok=True)
RESULTADOS = {'gpu': props.name, 'diffusers': diffusers.__version__}

## 3. Componentes de Stable Diffusion

Cargo cada componente por separado para poder inspeccionarlo y entrenar solo la U-Net. Los pesos del modelo base se cargan en media precisión (`float16`) para ahorrar memoria de la GPU.

In [ ]:
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler, StableDiffusionPipeline
from transformers import CLIPTextModel, CLIPTokenizer

MODELO = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
dtype = torch.float16

tokenizer = CLIPTokenizer.from_pretrained(MODELO, subfolder='tokenizer')
text_encoder = CLIPTextModel.from_pretrained(MODELO, subfolder='text_encoder', torch_dtype=dtype).to(dispositivo)
vae = AutoencoderKL.from_pretrained(MODELO, subfolder='vae', torch_dtype=dtype).to(dispositivo)
unet = UNet2DConditionModel.from_pretrained(MODELO, subfolder='unet', torch_dtype=dtype).to(dispositivo)
noise_scheduler = DDPMScheduler.from_pretrained(MODELO, subfolder='scheduler')

In [ ]:
contar = lambda m: sum(p.numel() for p in m.parameters())
tabla = {'VAE': contar(vae), 'U-Net': contar(unet), 'Codificador de texto CLIP': contar(text_encoder)}
for k, v in tabla.items():
    print(f'{k:28s} {v/1e6:8.1f} M parámetros')
print(f"{'Total':28s} {sum(tabla.values())/1e6:8.1f} M parámetros")
print('\nPasos de difusión del entrenamiento (T):', noise_scheduler.config.num_train_timesteps)
print('Factor de escala del VAE:', vae.config.scaling_factor)
RESULTADOS['parametros_M'] = {k: round(v / 1e6, 1) for k, v in tabla.items()}

ids = tokenizer('a photo of a butterfly in smthsn style', padding='max_length',
                max_length=tokenizer.model_max_length, return_tensors='pt').input_ids
with torch.no_grad():
    emb = text_encoder(ids.to(dispositivo))[0]
print('\nTokens del prompt:', tokenizer.convert_ids_to_tokens(ids[0][:12]))
print('Forma del embedding de texto:', tuple(emb.shape), '-> 77 tokens x 768 dimensiones')

## 4. El espacio latente y la difusión hacia adelante

Primero cargo los datos (sección 5 los prepara para el entrenamiento) para tener una imagen real con la cual experimentar.

In [ ]:
from datasets import load_dataset
ds = load_dataset('huggan/smithsonian_butterflies_subset', split='train')
print(ds)

In [ ]:
from torchvision import transforms
RES = 512
a_tensor = transforms.Compose([transforms.Resize(RES, interpolation=transforms.InterpolationMode.BILINEAR),
                               transforms.CenterCrop(RES), transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])

def a_pil(t):
    t = (t.detach().float().cpu().clamp(-1, 1) + 1) / 2
    return Image.fromarray((t.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8))

img = ds[0]['image'].convert('RGB')
x = a_tensor(img).unsqueeze(0).to(dispositivo, dtype)
with torch.no_grad():
    lat = vae.encode(x).latent_dist.sample() * vae.config.scaling_factor
    rec = vae.decode(lat / vae.config.scaling_factor).sample
print('Imagen:', tuple(x.shape), '=', x.numel(), 'valores')
print('Latente:', tuple(lat.shape), '=', lat.numel(), 'valores  -> compresión x%d' % (x.numel() // lat.numel()))
err = (rec.float() - x.float()).abs().mean().item()
print(f'Error absoluto medio de reconstrucción del VAE: {err:.4f} (en escala [-1,1])')
RESULTADOS['vae'] = {'forma_imagen': list(x.shape), 'forma_latente': list(lat.shape), 'error_reconstruccion': round(err, 4)}

fig, ax = plt.subplots(1, 6, figsize=(18, 3.4))
ax[0].imshow(a_pil(x[0])); ax[0].set_title('Original 512×512×3')
ax[1].imshow(a_pil(rec[0])); ax[1].set_title('Reconstrucción del VAE')
for c in range(4):
    ax[2 + c].imshow(lat[0, c].float().cpu(), cmap='viridis'); ax[2 + c].set_title(f'Latente canal {c} (64×64)')
for a in ax: a.axis('off')
plt.tight_layout(); plt.savefig('figuras/sd-01-espacio-latente-vae.png', dpi=130); plt.show()

In [ ]:
pasos_t = [0, 100, 250, 500, 750, 999]
ruido = torch.randn_like(lat)
fig, ax = plt.subplots(2, len(pasos_t), figsize=(18, 6))
for i, t in enumerate(pasos_t):
    lt = noise_scheduler.add_noise(lat.float(), ruido.float(), torch.tensor([t])).to(dtype)
    with torch.no_grad():
        dec = vae.decode(lt / vae.config.scaling_factor).sample
    ab = noise_scheduler.alphas_cumprod[t].item()
    ax[0, i].imshow(lt[0, :3].float().cpu().permute(1, 2, 0).add(3).div(6).clamp(0, 1)); ax[0, i].set_title(f't = {t}  (ᾱ = {ab:.3f})')
    ax[1, i].imshow(a_pil(dec[0]))
    ax[0, i].axis('off'); ax[1, i].axis('off')
ax[0, 0].set_ylabel('latente'); ax[1, 0].set_ylabel('decodificado')
plt.suptitle('Proceso de difusión hacia adelante en el espacio latente (fila 1: latente, fila 2: decodificado con el VAE)')
plt.tight_layout(); plt.savefig('figuras/sd-02-difusion-adelante.png', dpi=130); plt.show()

## 5. Preparación de los datos

El subconjunto de mariposas del Smithsonian tiene 1 000 fotografías de especímenes de museo (dominio público). Cada imagen se redimensiona a 512×512, se recorta al centro, se voltea horizontalmente al azar y se normaliza a $[-1,1]$. Como descripción uso una plantilla con la palabra disparadora `smthsn` y, si existe, el nombre común de la especie.

In [ ]:
DISPARADOR = 'smthsn'
def descripcion(ej):
    nombre = ej.get('name') if isinstance(ej.get('name'), str) else None
    base = f'a photo of a butterfly in {DISPARADOR} style'
    return f'{base}, {nombre.lower()}' if nombre and nombre.strip() else base

aumentar = transforms.Compose([transforms.Resize(RES, interpolation=transforms.InterpolationMode.BILINEAR),
                               transforms.CenterCrop(RES), transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])

def preparar(lote):
    filas = [dict(zip(lote.keys(), v)) for v in zip(*lote.values())]
    pix = [aumentar(f['image'].convert('RGB')) for f in filas]
    ids = tokenizer([descripcion(f) for f in filas], padding='max_length', truncation=True,
                    max_length=tokenizer.model_max_length, return_tensors='pt').input_ids
    return {'pixel_values': pix, 'input_ids': list(ids)}

ds_entren = ds.with_transform(preparar)
def juntar(ejemplos):
    return {'pixel_values': torch.stack([e['pixel_values'] for e in ejemplos]).float(),
            'input_ids': torch.stack([e['input_ids'] for e in ejemplos])}

print('Ejemplos de descripciones:')
for i in range(4):
    print(' -', descripcion(ds[i]))

fig, ax = plt.subplots(2, 6, figsize=(16, 5.6))
for a, i in zip(ax.ravel(), range(0, 600, 50)):
    a.imshow(ds[i]['image'].convert('RGB').resize((256, 256))); a.axis('off')
plt.suptitle('Muestras del conjunto de entrenamiento (Smithsonian butterflies)')
plt.tight_layout(); plt.savefig('figuras/sd-03-datos-entrenamiento.png', dpi=110); plt.show()

## 6. Generación antes del fine-tuning

Armo el *pipeline* de inferencia con **los mismos objetos** `unet`, `vae` y `text_encoder`. Así, cuando entrene la LoRA dentro de `unet`, el pipeline usará automáticamente los pesos ajustados y puedo comparar con exactamente las mismas semillas.

Uso cuatro prompts: dos de mariposas (el dominio entrenado) y dos de otros animales, para ver si el estilo se **generaliza** a objetos que no estaban en los datos.

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(MODELO, unet=unet, vae=vae, text_encoder=text_encoder, tokenizer=tokenizer,
                                               torch_dtype=dtype, safety_checker=None, requires_safety_checker=False).to(dispositivo)
pipe.set_progress_bar_config(disable=True)

PROMPTS = [f'a photo of a butterfly in {DISPARADOR} style',
           f'a blue butterfly with long tails in {DISPARADOR} style',
           f'a beetle in {DISPARADOR} style',
           f'a hummingbird in {DISPARADOR} style']
SEMILLAS = [0, 1]
PASOS, GUIA = 30, 7.5

def generar(escala=None):
    imgs = []
    kw = {} if escala is None else {'cross_attention_kwargs': {'scale': escala}}
    for p in PROMPTS:
        for s in SEMILLAS:
            g = torch.Generator(dispositivo).manual_seed(s)
            imgs.append(pipe(p, num_inference_steps=PASOS, guidance_scale=GUIA, generator=g, height=RES, width=RES, **kw).images[0])
    return imgs

def mostrar_rejilla(imgs, titulo, archivo):
    n = len(SEMILLAS)
    fig, ax = plt.subplots(n, len(PROMPTS), figsize=(4 * len(PROMPTS), 4 * n))
    ax = np.array(ax).reshape(n, len(PROMPTS))
    for j, p in enumerate(PROMPTS):
        for i in range(n):
            ax[i, j].imshow(imgs[j * n + i]); ax[i, j].axis('off')
        ax[0, j].set_title(p.replace(f' in {DISPARADOR} style', f'\nin {DISPARADOR} style'), fontsize=10)
    plt.suptitle(titulo); plt.tight_layout(); plt.savefig(archivo, dpi=110); plt.show()

t0 = time.time()
antes = generar()
t_gen = (time.time() - t0) / len(antes)
print(f'Tiempo por imagen ({PASOS} pasos, 512×512): {t_gen:.1f} s')
RESULTADOS['seg_por_imagen'] = round(t_gen, 2)
mostrar_rejilla(antes, 'ANTES del fine-tuning (Stable Diffusion v1.5 original)', 'figuras/sd-04-antes-finetuning.png')

## 7. Configuración de LoRA y entrenamiento

- Se congelan el VAE, el codificador de texto y todos los pesos originales de la U-Net.
- Se agregan adaptadores LoRA de rango $r=8$ en las proyecciones de atención (`to_q`, `to_k`, `to_v`, `to_out.0`).
- Los parámetros LoRA se mantienen en `float32` (para entrenar con estabilidad) mientras el resto del modelo sigue en `float16`, usando precisión mixta (`autocast` + `GradScaler`).
- *Gradient checkpointing* reduce el consumo de memoria recalculando activaciones durante la retropropagación.

| Hiperparámetro | Valor |
|---|---|
| Resolución | 512×512 |
| Lote × acumulación de gradiente | 2 × 2 (lote efectivo 4) |
| Pasos de optimización | 400 (≈ 1,6 épocas) |
| Optimizador | AdamW, $lr = 1\times10^{-4}$, decaimiento 0,01 |
| Programación de lr | constante con 20 pasos de calentamiento |
| Pérdida | MSE entre el ruido real y el predicho |

In [ ]:
from peft import LoraConfig
from peft.utils import get_peft_model_state_dict
from diffusers.training_utils import cast_training_params
from diffusers.utils import convert_state_dict_to_diffusers

RANGO = 8
vae.requires_grad_(False); text_encoder.requires_grad_(False); unet.requires_grad_(False)
config_lora = LoraConfig(r=RANGO, lora_alpha=RANGO, init_lora_weights='gaussian',
                         target_modules=['to_k', 'to_q', 'to_v', 'to_out.0'])
unet.add_adapter(config_lora)
cast_training_params(unet, dtype=torch.float32)        # solo los parámetros LoRA quedan en float32
unet.enable_gradient_checkpointing()

params_lora = [p for p in unet.parameters() if p.requires_grad]
n_lora, n_unet = sum(p.numel() for p in params_lora), contar(unet)
print(f'Parámetros entrenables (LoRA): {n_lora:,} de {n_unet:,} -> {100*n_lora/n_unet:.3f} % de la U-Net')
RESULTADOS['lora'] = {'rango': RANGO, 'parametros_entrenables': n_lora, 'porcentaje_unet': round(100 * n_lora / n_unet, 3)}

In [ ]:
LOTE, ACUM, PASOS_MAX, LR, CALENT = 2, 2, 400, 1e-4, 20

cargador = torch.utils.data.DataLoader(ds_entren, batch_size=LOTE, shuffle=True, collate_fn=juntar,
                                       num_workers=2, drop_last=True, generator=torch.Generator().manual_seed(SEMILLA))
optim = torch.optim.AdamW(params_lora, lr=LR, weight_decay=1e-2)
sched_lr = torch.optim.lr_scheduler.LambdaLR(optim, lambda s: min(1.0, (s + 1) / CALENT))
escalador = torch.amp.GradScaler('cuda')

perdidas, paso, micro = [], 0, 0
unet.train(); t0 = time.time()
while paso < PASOS_MAX:
    for lote in cargador:
        pix = lote['pixel_values'].to(dispositivo, dtype)
        with torch.no_grad():
            latentes = vae.encode(pix).latent_dist.sample() * vae.config.scaling_factor
            texto = text_encoder(lote['input_ids'].to(dispositivo))[0]
        ruido = torch.randn_like(latentes)
        t = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latentes.shape[0],), device=dispositivo).long()
        ruidosos = noise_scheduler.add_noise(latentes, ruido, t)

        with torch.autocast('cuda', dtype=torch.float16):
            pred = unet(ruidosos, t, encoder_hidden_states=texto).sample
        perdida = F.mse_loss(pred.float(), ruido.float()) / ACUM
        escalador.scale(perdida).backward()
        micro += 1
        perdidas.append(perdida.item() * ACUM)

        if micro % ACUM == 0:
            escalador.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(params_lora, 1.0)
            escalador.step(optim); escalador.update(); optim.zero_grad(set_to_none=True); sched_lr.step()
            paso += 1
            if paso % 25 == 0 or paso == 1:
                print(f'paso {paso:4d}/{PASOS_MAX} | pérdida media (últimos 50) {np.mean(perdidas[-50:]):.4f} | '
                      f'{time.time()-t0:6.0f} s | memoria GPU {torch.cuda.max_memory_allocated()/1024**3:.1f} GB')
            if paso >= PASOS_MAX:
                break

t_entren = time.time() - t0
unet.eval()
print(f'\nEntrenamiento terminado: {paso} pasos, {micro} micro-lotes, {t_entren/60:.1f} min')
RESULTADOS['entrenamiento'] = {'pasos': paso, 'minutos': round(t_entren / 60, 1),
                               'perdida_primeros_50': round(float(np.mean(perdidas[:50])), 4),
                               'perdida_ultimos_50': round(float(np.mean(perdidas[-50:])), 4),
                               'memoria_max_GB': round(torch.cuda.max_memory_allocated() / 1024**3, 2)}

In [ ]:
k = min(50, max(1, len(perdidas) // 4))
suave = np.convolve(perdidas, np.ones(k) / k, mode='valid')
plt.figure(figsize=(10, 3.8))
plt.plot(perdidas, alpha=.25, color='#4C72B0', label='pérdida por micro-lote')
plt.plot(np.arange(k - 1, len(perdidas)), suave, color='#C44E52', lw=2, label=f'media móvil ({k})')
plt.xlabel('micro-lote'); plt.ylabel('MSE del ruido'); plt.title('Pérdida de entrenamiento de la LoRA'); plt.legend()
plt.tight_layout(); plt.savefig('figuras/sd-05-perdida-entrenamiento.png', dpi=130); plt.show()
print(RESULTADOS['entrenamiento'])

## 8. Generación después del fine-tuning

Mismos prompts, mismas semillas y mismos hiperparámetros de muestreo que en la sección 6.

In [ ]:
despues = generar()
mostrar_rejilla(despues, 'DESPUÉS del fine-tuning con LoRA (400 pasos)', 'figuras/sd-06-despues-finetuning.png')

fig, ax = plt.subplots(2, len(PROMPTS), figsize=(4 * len(PROMPTS), 8.4))
for j, p in enumerate(PROMPTS):
    ax[0, j].imshow(antes[j * len(SEMILLAS)]); ax[1, j].imshow(despues[j * len(SEMILLAS)])
    ax[0, j].set_title(p.replace(f' in {DISPARADOR} style', ''), fontsize=11)
    ax[0, j].set_xticks([]); ax[0, j].set_yticks([]); ax[1, j].set_xticks([]); ax[1, j].set_yticks([])
ax[0, 0].set_ylabel('ANTES', fontsize=13); ax[1, 0].set_ylabel('DESPUÉS', fontsize=13)
plt.suptitle('Comparación con la misma semilla (0)'); plt.tight_layout()
plt.savefig('figuras/sd-07-comparacion-antes-despues.png', dpi=110); plt.show()

## 9. Pruebas

### 9.1 Intensidad del adaptador LoRA
La escala multiplica la actualización $BA$: con 0 se recupera el modelo original y con 1 se aplica la LoRA completa. Así se ve cómo el estilo aparece de forma gradual.

In [ ]:
escalas = [0.0, 0.25, 0.5, 0.75, 1.0]
p = PROMPTS[0]
fig, ax = plt.subplots(1, len(escalas), figsize=(4 * len(escalas), 4.4))
for a, e in zip(ax, escalas):
    g = torch.Generator(dispositivo).manual_seed(3)
    im = pipe(p, num_inference_steps=PASOS, guidance_scale=GUIA, generator=g, height=RES, width=RES,
              cross_attention_kwargs={'scale': e}).images[0]
    a.imshow(im); a.set_title(f'escala LoRA = {e}'); a.axis('off')
plt.suptitle(f'"{p}" con distinta intensidad del adaptador'); plt.tight_layout()
plt.savefig('figuras/sd-08-escala-lora.png', dpi=110); plt.show()

### 9.2 Métrica cuantitativa con CLIP
Para no quedarme solo con la impresión visual, uso el modelo CLIP (Radford et al., 2021) para medir:

- **Similitud con el dominio:** similitud coseno entre cada imagen generada y el *embedding* promedio de 200 imágenes reales del Smithsonian. Si el fine-tuning funcionó debe **subir**.
- **CLIP score con el texto:** similitud entre la imagen y el objeto pedido (*"a photo of a butterfly"*, *"a photo of a beetle"*, ...). Sirve para revisar que el modelo no "olvidó" qué objeto debía dibujar.

In [ ]:
from transformers import CLIPModel, CLIPProcessor
clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(dispositivo).eval()
proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

@torch.no_grad()
def emb_img(imgs):
    out = []
    for i in range(0, len(imgs), 32):
        e = clip.get_image_features(**proc(images=imgs[i:i + 32], return_tensors='pt').to(dispositivo))
        e = e if torch.is_tensor(e) else e.pooler_output
        out.append(F.normalize(e.float(), dim=-1))
    return torch.cat(out)

@torch.no_grad()
def emb_txt(txts):
    e = clip.get_text_features(**proc(text=txts, return_tensors='pt', padding=True).to(dispositivo))
    e = e if torch.is_tensor(e) else e.pooler_output
    return F.normalize(e.float(), dim=-1)

centro = F.normalize(emb_img([ds[i]['image'].convert('RGB') for i in range(200)]).mean(0), dim=0)
objetos = ['a photo of a butterfly', 'a photo of a butterfly', 'a photo of a beetle', 'a photo of a hummingbird']
txt = emb_txt(objetos)

print(f"{'Prompt':45s} | dominio antes | dominio después | CLIP-texto antes | CLIP-texto después")
metricas = []
for j, p in enumerate(PROMPTS):
    sl = slice(j * len(SEMILLAS), (j + 1) * len(SEMILLAS))
    ea, ed = emb_img(antes[sl]), emb_img(despues[sl])
    fila = {'prompt': p, 'dominio_antes': (ea @ centro).mean().item(), 'dominio_despues': (ed @ centro).mean().item(),
            'texto_antes': 100 * (ea @ txt[j]).mean().item(), 'texto_despues': 100 * (ed @ txt[j]).mean().item()}
    metricas.append(fila)
    print(f"{p:45s} | {fila['dominio_antes']:13.3f} | {fila['dominio_despues']:15.3f} | {fila['texto_antes']:16.1f} | {fila['texto_despues']:18.1f}")
RESULTADOS['clip'] = [{k: (round(v, 4) if isinstance(v, float) else v) for k, v in f.items()} for f in metricas]

### 9.3 Guardar la LoRA y volver a cargarla en un pipeline nuevo
El resultado del fine-tuning es solo el archivo de la LoRA. Lo guardo en formato `safetensors`, reviso su tamaño y lo cargo en un pipeline limpio para comprobar que reproduce el estilo.

In [ ]:
CARPETA_LORA = 'lora-smithsonian-mariposas'
estado = convert_state_dict_to_diffusers(get_peft_model_state_dict(unet))
StableDiffusionPipeline.save_lora_weights(save_directory=CARPETA_LORA, unet_lora_layers=estado, safe_serialization=True)
tam = sum(os.path.getsize(os.path.join(CARPETA_LORA, f)) for f in os.listdir(CARPETA_LORA)) / 1024**2
print(f'Archivos: {os.listdir(CARPETA_LORA)} | tamaño total: {tam:.1f} MB (el modelo completo pesa ~2 000 MB en fp16)')
RESULTADOS['lora']['tamano_MB'] = round(tam, 2)

pipe2 = StableDiffusionPipeline.from_pretrained(MODELO, torch_dtype=dtype, safety_checker=None, requires_safety_checker=False).to(dispositivo)
pipe2.set_progress_bar_config(disable=True)
pipe2.load_lora_weights(CARPETA_LORA)
g = torch.Generator(dispositivo).manual_seed(0)
recargada = pipe2(PROMPTS[0], num_inference_steps=PASOS, guidance_scale=GUIA, generator=g, height=RES, width=RES).images[0]

fig, ax = plt.subplots(1, 2, figsize=(9, 4.8))
ax[0].imshow(despues[0]); ax[0].set_title('LoRA recién entrenada'); ax[0].axis('off')
ax[1].imshow(recargada); ax[1].set_title('LoRA recargada desde el archivo'); ax[1].axis('off')
plt.tight_layout(); plt.savefig('figuras/sd-09-lora-recargada.png', dpi=110); plt.show()
dif = np.abs(np.asarray(despues[0], dtype=float) - np.asarray(recargada, dtype=float)).mean()
print(f'Diferencia media por píxel entre ambas imágenes: {dif:.2f} (de 255)')
RESULTADOS['diferencia_recarga'] = round(float(dif), 2)
del pipe2; torch.cuda.empty_cache()

### 9.4 Guardar resultados y descargar las figuras

In [ ]:
for i, im in enumerate(antes): im.save(f'figuras/sd-antes-{i}.png')
for i, im in enumerate(despues): im.save(f'figuras/sd-despues-{i}.png')
with open('figuras/resultados-sd.json', 'w', encoding='utf-8') as f:
    json.dump(RESULTADOS, f, ensure_ascii=False, indent=2)
print(json.dumps(RESULTADOS, ensure_ascii=False, indent=2))

import shutil
shutil.make_archive('resultados-stable-diffusion', 'zip', '.', 'figuras')
try:
    from google.colab import files
    files.download('resultados-stable-diffusion.zip')
except ImportError:
    print('Fuera de Colab: el archivo resultados-stable-diffusion.zip quedó en la carpeta actual.')

## 10. Análisis

**Qué aprende la LoRA.** Con solo 400 pasos y solo el 0,19 % de los parámetros de la U-Net entrenables, el modelo aprende a asociar la palabra `smthsn` con el estilo de fotografía de museo: ejemplar centrado, alas extendidas en vista superior y fondo claro y uniforme. Antes del ajuste, Stable Diffusion no conoce esa palabra y genera mariposas "artísticas" en entornos naturales.

**Generalización.** Los prompts de escarabajo y colibrí no estaban en los datos. Si el estilo se transfiere a ellos (fondo neutro, vista de catálogo), la LoRA aprendió un concepto de estilo y no solo a copiar mariposas. Si en cambio el colibrí empieza a parecer mariposa, es una señal de **sobreajuste** al único objeto del conjunto; esto se ve en la métrica CLIP-texto.

**La pérdida no mide la calidad.** La pérdida de difusión es muy ruidosa porque cada micro-lote usa pasos $t$ aleatorios: en $t$ cercanos a 999 predecir el ruido es fácil y en $t$ pequeños es difícil. Por eso la curva baja poco y las comparaciones visuales y con CLIP son más informativas.

**GAN vs. difusión.** La GAN del notebook 1 genera una imagen en un solo paso (milisegundos) pero es inestable de entrenar y difícil de condicionar con texto. Stable Diffusion necesita 30 pasos de eliminación de ruido por imagen, pero su entrenamiento es una regresión estable (MSE), no sufre colapso de modo y acepta condiciones en lenguaje natural.

**Limitaciones computacionales.** Según su ficha técnica, entrenar Stable Diffusion v1 desde cero costó alrededor de 150 000 horas de GPU A100. Con LoRA, precisión mixta y *gradient checkpointing* el ajuste cabe en una GPU T4 gratuita de Colab en unos 15-20 minutos.

### Referencias
- Ho, J., Jain, A. y Abbeel, P. (2020). *Denoising diffusion probabilistic models*. NeurIPS.
- Rombach, R. et al. (2022). *High-resolution image synthesis with latent diffusion models*. CVPR.
- Hu, E. J. et al. (2022). *LoRA: Low-rank adaptation of large language models*. ICLR.
- Ramesh, A. et al. (2022). *Hierarchical text-conditional image generation with CLIP latents*. arXiv:2204.06125.
- Radford, A. et al. (2021). *Learning transferable visual models from natural language supervision*. ICML.
- von Platen, P. et al. (2022). *Diffusers: State-of-the-art diffusion models*. GitHub.
- Latent Diffusion Models: The Architecture behind Stable Diffusion (lectura del curso).